<a href="https://colab.research.google.com/github/amna-png/ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amna-png/ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML-03 — Frame Your Lane as an ML Task

## FlyRank: CTR underperformance prioritization

This notebook frames the FlyRank lane as an ML problem before model training. The goal is to decide which content pages should be reviewed first using CTR relative to search-position expectations.

## 1. My lane as an ML task (type)

**Task type: Ranking.**

FlyRank is a ranking problem because the practical output is an ordered review queue: which pages should a content/SEO team inspect first? A simple low-CTR flag is not enough because CTR depends strongly on search position. The lane therefore focuses on identifying pages whose observed CTR is lower than the CTR expected for their position context, then ranking those pages by the size of the underperformance signal.

The decision-support output is not a claim that a page is definitely bad or that a change will cause improvement. It is a prioritized list for human review.

In [8]:

import pandas as pd

DATA_PATH = "content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique content IDs:", df["content_id"].nunique())


Rows: 30000
Columns: 44
Unique content IDs: 30000


## 2. Target or proxy

**Proxy target: CTR underperformance (`ctr_gap`).**

The current starter data does not contain a human-confirmed label such as `underperformer = 1/0`. Instead, I can define a directional proxy from the observed data: expected CTR is estimated within the existing `position_tier`, and `ctr_gap` is the difference between expected CTR and the page's observed CTR.

`ctr_gap = expected_ctr_by_position_tier - observed_ctr`

A larger positive gap means the observed CTR is further below the position-tier expectation, so the page has a stronger signal for review. This is a proxy for prioritization, not a causal or final-quality label. A later version can replace this proxy with an observed outcome or human-reviewed label.

In [9]:
# Sketch the proxy target using only fields present in the supplied dataset.
# Median CTR is used here as a simple, transparent position-tier expectation.
expected_ctr_by_tier = df.groupby("position_tier")["ctr"].median().rename("expected_ctr")
df["expected_ctr"] = df["position_tier"].map(expected_ctr_by_tier)
df["ctr_gap"] = df["expected_ctr"] - df["ctr"]

proxy_view = df[["content_id", "position_tier", "avg_position", "ctr", "expected_ctr", "ctr_gap"]].copy()
proxy_view.sort_values("ctr_gap", ascending=False).head(10)

,content_id,position_tier,avg_position,ctr,expected_ctr,ctr_gap
29983,content_6880eb215048,page_1,6.8,0.0,0.16,0.16
29977,content_c87291853cab,page_1,8.2,0.0,0.16,0.16
29973,content_4be930227848,page_1,3.5,0.0,0.16,0.16
25,content_033ae3e7aecf,page_1,7.2,0.0,0.16,0.16
29924,content_3d4004b21c6f,page_1,7.3,0.0,0.16,0.16
29913,content_a16cc7d0e1da,page_1,4.0,0.0,0.16,0.16
29906,content_07ce98c6085a,page_1,5.3,0.0,0.16,0.16
29902,content_e3de4ddde194,page_1,9.4,0.0,0.16,0.16
29886,content_04d69956e256,page_1,3.7,0.0,0.16,0.16
29871,content_b7ab61937a1c,page_1,7.0,0.0,0.16,0.16


## 3. Success metric
**Primary success metric: Recall@K.**

The real decision has a limited review capacity: a team may only be able to inspect the top K pages from the queue. Recall@K asks how many of the genuinely important underperformers are captured in those first K recommendations.

This metric matches the lane's priority: missing a page that really needs attention can leave potential improvement opportunities out of the review queue, while some false alarms can be reviewed and rejected by a human.

**Important limitation:** the supplied starter data does not contain an observed ground-truth review label, so a numeric Recall@K cannot honestly be reported yet. This notebook defines the metric to use once reviewed outcomes/labels are available; it does not invent a score.

In [10]:
# The current dataset has no human-reviewed ground-truth label.
# Check this explicitly rather than fabricating a Recall@K value.
possible_label_columns = [
    c for c in df.columns
    if any(term in c.lower() for term in ["label", "review", "underperform"])
]
print("Possible ground-truth label columns:", possible_label_columns)
print("Planned metric: Recall@K")

Possible ground-truth label columns: []
Planned metric: Recall@K


## 4. The unit of analysis, as a real dataframe

**One row = one content/page observation.**

The supplied lane dataset contains one row per `content_id`. The fields used for this framing include observed CTR, average search position, impressions, clicks, and position tier. Client identifiers are not needed for this notebook's display, so they are excluded from the working view.


In [11]:
# Show the actual unit of analysis from the supplied CSV.
unit_columns = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "position_tier"
]
unit_df = df[unit_columns].copy()

print("One row represents one content/page observation.")
print("Shape:", unit_df.shape)
unit_df.head(10)

One row represents one content/page observation.
Shape: (30000, 6)


,content_id,impressions_90d,clicks_90d,ctr,avg_position,position_tier
0,content_304f48230142,3803,29,0.76,10.6,striking
1,content_a1fb4e703a9e,15320,7,0.05,20.3,page_3_5
2,content_9aa793d4d895,12581,11,0.09,36.5,page_3_5
3,content_331d6c4de07b,11751,58,0.49,6.2,page_1
4,content_d99b7a2d90ca,19140,24,0.13,44.0,page_3_5
5,content_d4084a4bc775,3970,1,0.03,8.5,page_1
6,content_9a34b442b552,20,0,0.00,7.0,page_1
7,content_a63219c6e95a,1724,1,0.06,21.2,page_3_5
8,content_5e6c160719bc,32574,29,0.09,46.0,page_3_5
9,content_c27558df2b0c,1240,2,0.16,4.9,page_1


### Target/proxy sketch on the real dataframe

The derived columns below show what the prioritization signal looks like at row level: observed CTR, position-tier expected CTR, and the resulting CTR gap.

In [12]:
target_sketch = df[[
    "content_id", "position_tier", "avg_position",
    "ctr", "expected_ctr", "ctr_gap"
]].copy()
target_sketch.head(10)


,content_id,position_tier,avg_position,ctr,expected_ctr,ctr_gap
0,content_304f48230142,striking,10.6,0.76,0.11,-0.65
1,content_a1fb4e703a9e,page_3_5,20.3,0.05,0.03,-0.02
2,content_9aa793d4d895,page_3_5,36.5,0.09,0.03,-0.06
3,content_331d6c4de07b,page_1,6.2,0.49,0.16,-0.33
4,content_d99b7a2d90ca,page_3_5,44.0,0.13,0.03,-0.10
5,content_d4084a4bc775,page_1,8.5,0.03,0.16,0.13
6,content_9a34b442b552,page_1,7.0,0.00,0.16,0.16
7,content_a63219c6e95a,page_3_5,21.2,0.06,0.03,-0.03
8,content_5e6c160719bc,page_3_5,46.0,0.09,0.03,-0.06
9,content_c27558df2b0c,page_1,4.9,0.16,0.16,0.00


## 5. Why ML beats a fixed rule here

A fixed rule such as `IF CTR < 2% THEN flag the page` treats the same CTR as equally concerning at every search position. That is not a good fit for this lane because pages at different positions have different normal CTR expectations.

FlyRank instead needs a context-aware signal: compare observed CTR with an expected CTR for the page's search-position context, then rank the size of the underperformance. The current notebook uses a transparent position-tier median as a proxy baseline; a trained model can later learn a richer expected-CTR relationship from additional features.

This makes the problem more than a single if-statement: the useful output depends on the relationship between position/context and CTR, and the final output is an ordered review queue rather than a binary flag.

**Real content action:** a marketing/SEO team can use the ranked queue to decide which pages to investigate first, while keeping the final review decision with a human.

In [13]:
# Illustrate the difference between a fixed threshold and the position-aware proxy.
fixed_rule_rate = (df["ctr"] < 2).mean()
position_aware_rate = (df["ctr_gap"] > 0).mean()

print(f"Rows flagged by fixed CTR < 2% rule: {fixed_rule_rate:.1%}")
print(f"Rows with positive position-aware CTR gap: {position_aware_rate:.1%}")
print("These are descriptive checks only; they are not model-performance scores.")


Rows flagged by fixed CTR < 2% rule: 97.4%
Rows with positive position-aware CTR gap: 43.6%
These are descriptive checks only; they are not model-performance scores.




```
# This is formatted as code
```

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.